### 子图（Subgraph）

**子图就是一张编译好的 `CompiledStateGraph`，被当成一个节点塞进另一张图。** 没有别的魔法：`add_node("名字", 已编译的图)` 就完成了「调用」。

它解决什么问题：

- **复用**：Agent Loop、检索-重排、审批流这些模式会在多处出现，抽成子图后只写一遍。
- **分层**：把一张几十个节点的大图拆成几块，每块单独 `stream` / 单独测试。
- **隔离**：子图可以有自己的私有状态字段，不必污染父图。

本节分两半：**怎么调**（状态怎么流动、返回什么），以及**配checkpointer 会发生什么**（这才是容易踩坑的地方）。

#### 先搭两张最小的图

父子图都声明 `log`（累积语义）和 `answer`（覆盖语义），另外父图多一个 `question`。注意 `question` **不在子图的 state 里**——这正是后面要讲的「私有 vs 共享」。

In [1]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class ParentState(TypedDict):
    question: str          # 父图私有：子图看不到，也回不来
    answer: str            # 共享键：覆盖语义
    log: Annotated[list[str], operator.add]   # 共享键：累积语义


class SubState(TypedDict):
    answer: str
    log: Annotated[list[str], operator.add]


INITIAL = {"question": "LangGraph 怎么用？", "answer": "", "log": []}


def prepare(state: ParentState):
    """父图节点：只碰父图自己的字段。"""
    print("  [父] prepare 看到 question =", state["question"])
    return {"log": ["prepare"]}


def draft(state: SubState):
    """子图节点 1：只能看到 SubState 里声明过的键。"""
    print("  [子] draft 看到键 =", sorted(state))
    return {"answer": state["answer"] + "第一稿;", "log": ["draft"]}


def polish(state: SubState):
    print("  [子] polish 看到 answer =", state["answer"])
    return {"answer": state["answer"] + "定稿。", "log": ["polish"]}

### 1. 调用方式一：编译后当节点（推荐）

先 `compile()` 子图，再传给父图的 `add_node()`。父图跑到这个节点时，会**自动以「父图当前 state」为入参调用整张子图**，等它跑完，把结果并回父图。

> ⚠️ 必须**先 compile**。直接把没编译的 `StateGraph` builder 传给 `add_node()` 会报
> `TypeError: Expected a Runnable, callable or dict ... got an unsupported type: StateGraph`。

In [2]:
sub = StateGraph(SubState)
sub.add_node("draft", draft)
sub.add_node("polish", polish)
sub.add_edge(START, "draft")
sub.add_edge("draft", "polish")
sub.add_edge("polish", END)

subgraph = sub.compile()          # ← 必须先编译


parent = StateGraph(ParentState)
parent.add_node("prepare", prepare)
parent.add_node("sub_node", subgraph)      # ← 编译好的图直接当节点
parent.add_node("after", lambda s: {"log": ["after"]})
parent.add_edge(START, "prepare")
parent.add_edge("prepare", "sub_node")
parent.add_edge("sub_node", "after")
parent.add_edge("after", END)

graph = parent.compile()

print("result =", graph.invoke(dict(INITIAL)))

  [父] prepare 看到 question = LangGraph 怎么用？
  [子] draft 看到键 = ['answer', 'log']
  [子] polish 看到 answer = 第一稿;
result = {'question': 'LangGraph 怎么用？', 'answer': '第一稿;定稿。', 'log': ['prepare', 'prepare', 'draft', 'polish', 'after']}


#### 子图节点看到什么

三条规则，都能在上面的 `print` 里验证：

| 规则 | 说明 |
| --- | --- |
| **同名才可见** | 子图节点收到的 state 就是「父图 state 按子图 schema 过滤」的结果。`SubState` 没声明 `question`，子图里就看不到它。 |
| **同名必须同类型** | 共享键在父子两边都要有 compatible 的 reducer。否则子图算出来的值会在父图再被归约一次（下面详说）。 |
| **私有键不回传** | 子图里新增的字段（如果子图 state 有父图没有的键）不会污染父图。父图 schema 里没有的键，写了也被丢掉。 |

In [3]:
# 子图私有字段：父图 schema 里没有 steps，子图写了也回不来
class SubWithPrivate(SubState):
    steps: Annotated[list[str], operator.add]      # 父图 ParentState 没有这个键


sub2 = StateGraph(SubWithPrivate)
sub2.add_node("draft", lambda s: {"answer": s["answer"] + "x", "steps": ["draft"]})
sub2.add_edge(START, "draft")
sub2.add_edge("draft", END)

parent2 = StateGraph(ParentState)
parent2.add_node("sub_node", sub2.compile())
parent2.add_edge(START, "sub_node")
parent2.add_edge("sub_node", END)

out = parent2.compile().invoke(dict(INITIAL))
print("父图结果 =", out)
print("父图里有 steps 吗？", "steps" in out, "  ← 子图私有键被丢弃")

父图结果 = {'question': 'LangGraph 怎么用？', 'answer': 'x', 'log': []}
父图里有 steps 吗？ False   ← 子图私有键被丢弃


### 2. ⚠️ 最大的坑：共享键的 reducer 会被**归约两次**

当 `log` 这类累积键同时存在于父子 state 时，子图节点的返回值是**子图的完整 state**（不是子图的增量！），父图再用自己的 reducer 归约一遍，于是父图传进去的值被重复计算。

In [4]:
for chunk in graph.stream(dict(INITIAL), stream_mode="updates"):
    print(" ", chunk)

  [父] prepare 看到 question = LangGraph 怎么用？
  {'prepare': {'log': ['prepare']}}
  [子] draft 看到键 = ['answer', 'log']
  [子] polish 看到 answer = 第一稿;
  {'sub_node': {'answer': '第一稿;定稿。', 'log': ['prepare', 'draft', 'polish']}}
  {'after': {'log': ['after']}}


看清楚 `sub_node` 那一行：子图内部算出的 `log` 是 `['prepare', 'draft', 'polish']`（含父图传进去的 `prepare`），父图再拿它跟自己的 `['prepare']` 归约，结果就变成 `['prepare', 'prepare', 'draft', 'polish']`——`prepare` 出现了两次。

三种修法：

| 方案 | 做法 | 适用 |
| --- | --- | --- |
| A. 子图不声明该键 | 子图私有化 `log`，只共享 `answer` | 最省事，推荐 |
| B. 父子都用覆盖语义 | 两边 `log` 都不带 reducer | 该键本来就该「整体替换」 |
| C. 包装节点显式映射 | 用普通函数当节点，手动 `invoke` 并只回传需要的键 | 需要精细控制出入参 |

In [5]:
# 方案 A：子图把 log 私有化，只共享 answer
class SubClean(TypedDict):
    answer: str                                  # 唯一共享键，覆盖语义
    steps: Annotated[list[str], operator.add]    # 子图内部用，不回传


clean = StateGraph(SubClean)
clean.add_node("draft", lambda s: {"answer": s["answer"] + "第一稿;", "steps": ["draft"]})
clean.add_node("polish", lambda s: {"answer": s["answer"] + "定稿。", "steps": ["polish"]})
clean.add_edge(START, "draft")
clean.add_edge("draft", "polish")
clean.add_edge("polish", END)

parent_a = StateGraph(ParentState)
parent_a.add_node("prepare", prepare)
parent_a.add_node("sub_node", clean.compile())
parent_a.add_edge(START, "prepare")
parent_a.add_edge("prepare", "sub_node")
parent_a.add_edge("sub_node", END)

graph_a = parent_a.compile()
for chunk in graph_a.stream(dict(INITIAL), stream_mode="updates"):
    print(" ", chunk)
print("result =", graph_a.invoke(dict(INITIAL)))

  [父] prepare 看到 question = LangGraph 怎么用？
  {'prepare': {'log': ['prepare']}}
  {'sub_node': {'answer': '第一稿;定稿。'}}
  [父] prepare 看到 question = LangGraph 怎么用？
result = {'question': 'LangGraph 怎么用？', 'answer': '第一稿;定稿。', 'log': ['prepare']}


In [6]:
# 方案 C：包装节点，自己控制出入参
class SubIn(TypedDict):
    answer: str


class SubOut(TypedDict):
    answer: str


inner = StateGraph(SubIn, output_schema=SubOut)
inner.add_node("draft", lambda s: {"answer": s["answer"] + "第一稿;"})
inner.add_edge(START, "draft")
inner.add_edge("draft", END)
inner_graph = inner.compile()


def call_sub(state: ParentState):
    """手动调子图，只把需要的键传进去、把需要的键带回来。"""
    result = inner_graph.invoke({"answer": state["answer"]})
    return {"answer": result["answer"]}      # 故意不返回 log


parent_c = StateGraph(ParentState)
parent_c.add_node("prepare", prepare)
parent_c.add_node("sub_node", call_sub)
parent_c.add_edge(START, "prepare")
parent_c.add_edge("prepare", "sub_node")
parent_c.add_edge("sub_node", END)

graph_c = parent_c.compile()
for chunk in graph_c.stream(dict(INITIAL), stream_mode="updates"):
    print(" ", chunk)
print("result =", graph_c.invoke(dict(INITIAL)))

  [父] prepare 看到 question = LangGraph 怎么用？
  {'prepare': {'log': ['prepare']}}
  {'sub_node': {'answer': '第一稿;'}}
  [父] prepare 看到 question = LangGraph 怎么用？
result = {'question': 'LangGraph 怎么用？', 'answer': '第一稿;', 'log': ['prepare']}


### 3. 调用方式二：节点内部手动 `invoke`

方案 C 那种写法。它和「当节点」的区别在于**父图完全看不见子图内部**：

| | 当节点（`add_node("x", subgraph)`） | 节点内 `invoke` |
| --- | --- | --- |
| 子图内部节点在 `updates` 里 | 看得到（父节点名下） | 看不到 |
| 子图的 `interrupt()` 能否被父图捕获 | ✅ 能 | ❌ 不能 |
| 子图能否用自己的 checkpointer | ❌ 见下节 | ✅ 能 |
| 状态映射 | 按 schema 自动过滤 | 完全手写 |
| 适合 | 正常复用 | 需要跨越边界的独立编排 |

In [7]:
# 当节点：子图内部的 updates 会被合并到父节点名下
print("--- 当节点，subgraphs=True ---")
for namespace, chunk in graph.stream(dict(INITIAL), stream_mode="updates", subgraphs=True):
    print("  ", namespace or "顶层", "->", chunk)

--- 当节点，subgraphs=True ---
  [父] prepare 看到 question = LangGraph 怎么用？
   顶层 -> {'prepare': {'log': ['prepare']}}
  [子] draft 看到键 = ['answer', 'log']
  [子] polish 看到 answer = 第一稿;
   ('sub_node:36c94af4-17f2-5721-24de-be423114218a',) -> {'draft': {'answer': '第一稿;', 'log': ['draft']}}
   ('sub_node:36c94af4-17f2-5721-24de-be423114218a',) -> {'polish': {'answer': '第一稿;定稿。', 'log': ['polish']}}
   顶层 -> {'sub_node': {'answer': '第一稿;定稿。', 'log': ['prepare', 'draft', 'polish']}}
   顶层 -> {'after': {'log': ['after']}}


namespace 是个元组，空元组 `()` 表示顶层图，子图节点是 `("sub_node:<task_id>",)`——**带一个 task id**，因为同一次运行里可能多次调用同一个子图节点，每次都是独立的一份状态。

### 4. checkpointer：父图挂了，子图自动跟着挂

这是本节的重点。**子图不需要自己 `compile(checkpointer=...)`**：只要父图编译时挂了 checkpointer，子图作为节点执行时会自动用同一个 saver，并把自己的 checkpoint 写进**独立的 `checkpoint_ns` 命名空间**。

三层嵌套时命名空间用 `|` 拼接：`顶层 → sub_node:<id> → inner:<id>`。

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

saver = InMemorySaver()
graph_cp = parent.compile(checkpointer=saver)     # 只有父图挂了 saver

config = {"configurable": {"thread_id": "demo-1"}}
result = graph_cp.invoke(dict(INITIAL), config)
print("result =", result)
print()
print("--- saver 里出现了几个命名空间 ---")
for ns, checkpoints in saver.storage["demo-1"].items():
    print(f"  thread_id=demo-1  checkpoint_ns={ns!r}  共 {len(checkpoints)} 个快照")

  [父] prepare 看到 question = LangGraph 怎么用？
  [子] draft 看到键 = ['answer', 'log']
  [子] polish 看到 answer = 第一稿;
result = {'question': 'LangGraph 怎么用？', 'answer': '第一稿;定稿。', 'log': ['prepare', 'prepare', 'draft', 'polish', 'after']}

--- saver 里出现了几个命名空间 ---
  thread_id=demo-1  checkpoint_ns=''  共 5 个快照
  thread_id=demo-1  checkpoint_ns='sub_node:486d226c-e68d-705c-b3db-f59f8f032e47'  共 4 个快照


#### 按命名空间分别读状态

父图的 `get_state(config)` 只看得到顶层。要看子图内部，得显式带上 `checkpoint_ns`。这样就能问「子图跑到哪一步了」。

In [9]:
snapshot = graph_cp.get_state(config)
print("顶层 next =", snapshot.next)
print("顶层 values =", snapshot.values)
print()

for ns in saver.storage["demo-1"]:
    sub_config = {"configurable": {"thread_id": "demo-1", "checkpoint_ns": ns}}
    snap = graph_cp.get_state(sub_config)
    print(f"  ns={ns[:36]!r:40} next={str(snap.next):<14} values={snap.values}")

顶层 next = ()
顶层 values = {'question': 'LangGraph 怎么用？', 'answer': '第一稿;定稿。', 'log': ['prepare', 'prepare', 'draft', 'polish', 'after']}

  ns=''                                       next=()             values={'question': 'LangGraph 怎么用？', 'answer': '第一稿;定稿。', 'log': ['prepare', 'prepare', 'draft', 'polish', 'after']}
  ns='sub_node:486d226c-e68d-705c-b3db-f59'   next=()             values={'answer': '第一稿;定稿。', 'log': ['prepare', 'draft', 'polish']}


#### ⚠️ 子图自己挂的 checkpointer 会被忽略

如果子图 `compile(checkpointer=sub_saver)` 而父图也挂了 saver，**生效的是父图那个**，`sub_saver` 一个快照都不会有。原因是子图节点执行时的 config 由父图传入，父图的 checkpointer 覆盖了子图的。

想让子图彻底自治，只能用「节点内手动 `invoke`」那种写法。

In [10]:
sub_saver = InMemorySaver()
sub_own = StateGraph(SubClean)
sub_own.add_node("draft", lambda s: {"answer": s["answer"] + "x", "steps": ["draft"]})
sub_own.add_edge(START, "draft")
sub_own.add_edge("draft", END)

autonomous = sub_own.compile(checkpointer=sub_saver)     # 子图自带 saver

parent_auto = StateGraph(ParentState)
parent_auto.add_node("sub_node", autonomous)
parent_auto.add_edge(START, "sub_node")
parent_auto.add_edge("sub_node", END)

parent_saver = InMemorySaver()
parent_saver2 = parent_auto.compile(checkpointer=parent_saver)
cfg2 = {"configurable": {"thread_id": "t"}}
parent_saver2.invoke(dict(INITIAL), cfg2)

print("父 saver 的命名空间 =", list(parent_saver.storage["t"]))
print("子 saver 的内容       =", dict(sub_saver.storage) or "空  ← 被父图 checkpointer 覆盖了")

父 saver 的命名空间 = ['', 'sub_node:144b3a13-39bf-48a7-65e1-bdb556a8e474']
子 saver 的内容       = 空  ← 被父图 checkpointer 覆盖了


### 5. checkpointer + 子图：断点续跑与中断

挂上 checkpointer 后，子图内部的中断和失败同样能被父图接住。这是子图最有价值的用法之一——把「需要人工审批」的那几步打包成子图，父图只看到一个会暂停的节点。

In [11]:
from langgraph.types import Command, interrupt


def approve(state: SubState):
    """子图里的人工审批点。"""
    decision = interrupt(f"是否批准？当前答案：{state['answer'] or '（空）'}")
    return {"answer": state["answer"] + f"[{decision}]", "log": ["approved"]}


def revise(state: SubState):
    return {"answer": state["answer"] + "已按意见修订。", "log": ["revise"]}


review = StateGraph(SubState)
review.add_node("draft", draft)
review.add_node("approve", approve)
review.add_node("revise", revise)
review.add_edge(START, "draft")
review.add_edge("draft", "approve")
review.add_edge("approve", "revise")
review.add_edge("revise", END)

parent_h = StateGraph(ParentState)
parent_h.add_node("prepare", prepare)
parent_h.add_node("review_node", review.compile())
parent_h.add_node("after", lambda s: {"log": ["after"]})
parent_h.add_edge(START, "prepare")
parent_h.add_edge("prepare", "review_node")
parent_h.add_edge("review_node", "after")
parent_h.add_edge("after", END)

hitl = parent_h.compile(checkpointer=InMemorySaver())
hitl_config = {"configurable": {"thread_id": "approval-1"}}

In [12]:
# 第一次跑到 interrupt() 就停住
first = hitl.invoke(dict(INITIAL), hitl_config)
print("第一次返回 =", first)
print()
print("父图视角：next =", hitl.get_state(hitl_config).next)
print("子图视角：")
for ns in hitl.checkpointer.storage["approval-1"]:
    snap = hitl.get_state({"configurable": {"thread_id": "approval-1", "checkpoint_ns": ns}})
    print(f"  ns={ns[:36]!r:40} next={str(snap.next):<20} values={snap.values}")

  [父] prepare 看到 question = LangGraph 怎么用？
  [子] draft 看到键 = ['answer', 'log']
第一次返回 = {'question': 'LangGraph 怎么用？', 'answer': '', 'log': ['prepare'], '__interrupt__': [Interrupt(value='是否批准？当前答案：第一稿;', id='8602d491c56ac4babb3309cf6b880dba')]}

父图视角：next = ('review_node',)
子图视角：
  ns=''                                       next=('review_node',)     values={'question': 'LangGraph 怎么用？', 'answer': '', 'log': ['prepare']}
  ns='review_node:d06f983d-40dc-3b9e-7d96-'   next=('approve',)         values={'answer': '第一稿;', 'log': ['prepare', 'draft']}


In [13]:
# 恢复：Command(resume=...) 的值会被 interrupt() 拿到
second = hitl.invoke(Command(resume="批准"), hitl_config)
print("恢复后 =", second)
print()
print("最终 answer =", hitl.get_state(hitl_config).values["answer"])

恢复后 = {'question': 'LangGraph 怎么用？', 'answer': '第一稿;[批准]已按意见修订。', 'log': ['prepare', 'prepare', 'draft', 'approved', 'revise', 'after']}

最终 answer = 第一稿;[批准]已按意见修订。


关键点：**父图不需要知道子图内部有几个节点**。`hitl.get_state(...)` 看到的 `next` 就是 `('review_node',)`——那个还没跑完的子图节点。想知道子图内部卡在哪一步，就带 `checkpoint_ns` 去查。

#### 静态断点 `interrupt_before` 在子图上的行为

想在子图的某个节点前停下，**不能**在父图的 `invoke(interrupt_before=...)` 里写子图内部的节点名——父图只认自己这层的节点名。实测：

| 写法 | 结果 |
| --- | --- |
| 父图 `interrupt_before=["sub_node"]` | ✅ 生效（父图这层的节点名） |
| 父图 `interrupt_before=["draft"]`（子图内部节点名） | ❌ 静默失效，图一路跑到底 |
| 子图 `compile(interrupt_before=["polish"])` | ✅ 生效（要在子图自己 compile 时写死） |
| `interrupt_before="*"` | ✅ 顶层逐节点单步；停在子图节点时是整个子图一起停 |

In [14]:
leaf_ib = StateGraph(SubState)
leaf_ib.add_node("draft", draft)
leaf_ib.add_node("polish", polish)
leaf_ib.add_edge(START, "draft")
leaf_ib.add_edge("draft", "polish")
leaf_ib.add_edge("polish", END)

top_ib = StateGraph(ParentState)
top_ib.add_node("prepare", prepare)
top_ib.add_node("sub_node", leaf_ib.compile())
top_ib.add_node("after", lambda s: {"log": ["after"]})
top_ib.add_edge(START, "prepare")
top_ib.add_edge("prepare", "sub_node")
top_ib.add_edge("sub_node", "after")
top_ib.add_edge("after", END)

ib_graph = top_ib.compile(checkpointer=InMemorySaver())

# ❌ 父图传子图内部节点名：无效，一路跑完
cfg_bad = {"configurable": {"thread_id": "bad"}}
print("父图 interrupt_before=['polish']:")
print("   跑到 =", ib_graph.invoke(dict(INITIAL), cfg_bad, interrupt_before=["polish"])["log"])
print("   next =", ib_graph.get_state(cfg_bad).next)

# ✅ 父图这层的节点名：有效
cfg_ok = {"configurable": {"thread_id": "ok"}}
print("\n父图 interrupt_before=['sub_node']:")
print("   跑到 =", ib_graph.invoke(dict(INITIAL), cfg_ok, interrupt_before=["sub_node"])["log"])
print("   next =", ib_graph.get_state(cfg_ok).next)

父图 interrupt_before=['polish']:
  [父] prepare 看到 question = LangGraph 怎么用？
  [子] draft 看到键 = ['answer', 'log']
  [子] polish 看到 answer = 第一稿;
   跑到 = ['prepare', 'prepare', 'draft', 'polish', 'after']
   next = ()

父图 interrupt_before=['sub_node']:
  [父] prepare 看到 question = LangGraph 怎么用？
   跑到 = ['prepare']
   next = ('sub_node',)


In [15]:
# ✅ 子图自己 compile 时写死断点
leaf_fixed = StateGraph(SubState)
leaf_fixed.add_node("draft", draft)
leaf_fixed.add_node("polish", polish)
leaf_fixed.add_edge(START, "draft")
leaf_fixed.add_edge("draft", "polish")
leaf_fixed.add_edge("polish", END)
leaf_fixed_g = leaf_fixed.compile(interrupt_before=["polish"])   # 写死在子图上

top_fixed = StateGraph(ParentState)
top_fixed.add_node("prepare", prepare)
top_fixed.add_node("sub_node", leaf_fixed_g)
top_fixed.add_node("after", lambda s: {"log": ["after"]})
top_fixed.add_edge(START, "prepare")
top_fixed.add_edge("prepare", "sub_node")
top_fixed.add_edge("sub_node", "after")
top_fixed.add_edge("after", END)

fixed_graph = top_fixed.compile(checkpointer=InMemorySaver())
cfg_fixed = {"configurable": {"thread_id": "fixed"}}

print("跑到 =", fixed_graph.invoke(dict(INITIAL), cfg_fixed)["log"])
print("顶层 next =", fixed_graph.get_state(cfg_fixed).next)
for ns in fixed_graph.checkpointer.storage["fixed"]:
    snap = fixed_graph.get_state({"configurable": {"thread_id": "fixed", "checkpoint_ns": ns}})
    if ns:
        print(f"  子图内部 next = {snap.next}   ← 卡在 polish 前")
print("续跑 =", fixed_graph.invoke(None, cfg_fixed)["log"])

  [父] prepare 看到 question = LangGraph 怎么用？
  [子] draft 看到键 = ['answer', 'log']
跑到 = ['prepare']
顶层 next = ('sub_node',)
  子图内部 next = ('polish',)   ← 卡在 polish 前
  [子] polish 看到 answer = 第一稿;
续跑 = ['prepare', 'prepare', 'draft', 'polish', 'after']


### 6. 完整例子：一个带审批的「检索 → 成稿」流程

把上面的知识合起来：父子分层、状态只共享该共享的、checkpointer 挂在父图、中断从子图冒出来、子图被**打回后重跑**形成闭环。跑完再从 `get_state_history` 回看整条执行轨迹。

这个例子不调大模型，纯靠 `interrupt()` 演示，所以可以离线跑。

字段分三类，注释就是设计说明：

- **共享**：`query`（父图提问、子图拿去检索）、`retrieved`（子图累积检索结果）、`draft`、`approved`——父子 schema 里同名同类型。
- **私有**：`trail` 只在父图 `OverallState` 里声明，子图看不到也回不来。

In [16]:
# ---- 子图：检索 + 起草 ----
class WriterState(TypedDict):
    query: str
    retrieved: Annotated[list[str], operator.add]
    draft: str
    approved: bool


def retrieve(state: WriterState):
    """模拟检索。注意只往 retrieved 里加，子图内部不会碰 trail。"""
    docs = [f"《{state['query']}》相关文档 1", f"《{state['query']}》相关文档 2"]
    return {"retrieved": docs}


def compose(state: WriterState):
    """把检索结果拼成草稿。这里不调 LLM，保持示例可离线运行。"""
    body = "\n".join(f"- {d}" for d in state["retrieved"])
    return {"draft": f"关于「{state['query']}」：\n{body}"}


def gate(state: WriterState):
    """人工审批门。interrupt() 抛出的 Interrupt 会被父图接住。"""
    decision = interrupt({
        "question": "是否批准这份草稿？",
        "draft_preview": state["draft"][:40],
    })
    return {"approved": decision == "批准"}

In [17]:
writer = StateGraph(WriterState)
writer.add_node("retrieve", retrieve)
writer.add_node("compose", compose)
writer.add_node("gate", gate)
writer.add_edge(START, "retrieve")
writer.add_edge("retrieve", "compose")
writer.add_edge("compose", "gate")
writer.add_edge("gate", END)
writer_g = writer.compile()


class OverallState(TypedDict):
    query: str
    # retrieved / draft / approved 是子图与父图共享的键
    retrieved: Annotated[list[str], operator.add]
    draft: str
    approved: bool
    trail: Annotated[list[str], operator.add]   # 私有：父图专用


def start(state: OverallState):
    return {"trail": ["start"]}


def publish(state: OverallState):
    # 只在 approved 为 True 时往下走，否则打回重写
    return {"trail": ["publish" if state["approved"] else "reject"]}


def rewrite(state: OverallState):
    """打回：清掉 approved，让条件边回到 start 形成闭环。"""
    return {"trail": ["rewrite"], "approved": False}


pipeline = StateGraph(OverallState)
pipeline.add_node("start", start)
pipeline.add_node("writer", writer_g)          # ← 子图当节点
pipeline.add_node("publish", publish)
pipeline.add_node("rewrite", rewrite)
pipeline.add_edge(START, "start")
pipeline.add_edge("start", "writer")
pipeline.add_conditional_edges(
    "writer",
    lambda s: "publish" if s["approved"] else "rewrite",
    {"publish": "publish", "rewrite": "rewrite"},
)
pipeline.add_edge("rewrite", "start")          # 打回重跑，形成闭环
pipeline.add_edge("publish", END)

app = pipeline.compile(checkpointer=InMemorySaver())   # 只有父图挂 saver
app_config = {"configurable": {"thread_id": "writer-demo"}}

In [18]:
# 第一轮：跑到子图里的 gate 就暂停
res = app.invoke({"query": "LangGraph 的流式输出", "trail": []}, app_config)
print("第一轮返回 =", res)
print("__interrupt__ =", res.get("__interrupt__"))
print()
print("顶层 next =", app.get_state(app_config).next, " ← 父图只知道 writer 还没完")

第一轮返回 = {'query': 'LangGraph 的流式输出', 'retrieved': [], 'trail': ['start'], '__interrupt__': [Interrupt(value={'question': '是否批准这份草稿？', 'draft_preview': '关于「LangGraph 的流式输出」：\n- 《LangGraph 的流式输出》'}, id='56421f17c9fa9319f1f968eaa79fe491')]}
__interrupt__ = [Interrupt(value={'question': '是否批准这份草稿？', 'draft_preview': '关于「LangGraph 的流式输出」：\n- 《LangGraph 的流式输出》'}, id='56421f17c9fa9319f1f968eaa79fe491')]

顶层 next = ('writer',)  ← 父图只知道 writer 还没完


In [19]:
# 看看子图内部卡在哪
for ns in app.checkpointer.storage["writer-demo"]:
    snap = app.get_state({"configurable": {"thread_id": "writer-demo", "checkpoint_ns": ns}})
    print(f"  ns={ns[:44]!r:48} next={str(snap.next):<12}")
    print(f"      retrieved={snap.values.get('retrieved')}")

  ns=''                                               next=('writer',) 
      retrieved=[]
  ns='writer:fc02aaf6-d2ab-3c21-723e-4b6a51ab3c88'    next=('gate',)   
      retrieved=['《LangGraph 的流式输出》相关文档 1', '《LangGraph 的流式输出》相关文档 2']


`next` 是 `('gate',)`，且 `retrieved` 已经有两条结果——说明子图的 `retrieve` / `compose` 都跑完了，就卡在人工门上。这就是分层的价值：**审批逻辑被封在子图里，父图只处理「通过 / 打回」两条边**。

顺带注意 `retrieved` 在子图里也带了 `operator.add`——它是共享键，所以每轮重跑都会再追加两条。这正是上一节那个「共享累积键」的另一个副作用：重试 / 回环时数据会越堆越多。真要这样用，把 `retrieved` 也改成覆盖语义（子图每次返回完整列表）更安全。

In [20]:
# 打回：子图会重跑，父图走 rewrite 分支
res2 = app.invoke(Command(resume="打回"), app_config)
print("打回后 =", res2)
print("顶层 next =", app.get_state(app_config).next)

打回后 = {'query': 'LangGraph 的流式输出', 'retrieved': ['《LangGraph 的流式输出》相关文档 1', '《LangGraph 的流式输出》相关文档 2'], 'draft': '关于「LangGraph 的流式输出」：\n- 《LangGraph 的流式输出》相关文档 1\n- 《LangGraph 的流式输出》相关文档 2', 'approved': False, 'trail': ['start', 'rewrite', 'start'], '__interrupt__': [Interrupt(value={'question': '是否批准这份草稿？', 'draft_preview': '关于「LangGraph 的流式输出」：\n- 《LangGraph 的流式输出》'}, id='93a4cc972e5e9405db9967304e2322c5')]}
顶层 next = ('writer',)


In [21]:
# 再来一轮，这次批准
res3 = app.invoke(Command(resume="批准"), app_config)
print("批准后 =", res3)
print()
final = app.get_state(app_config)
print("approved =", final.values["approved"])
print("trail    =", final.values["trail"])

批准后 = {'query': 'LangGraph 的流式输出', 'retrieved': ['《LangGraph 的流式输出》相关文档 1', '《LangGraph 的流式输出》相关文档 2', '《LangGraph 的流式输出》相关文档 1', '《LangGraph 的流式输出》相关文档 2', '《LangGraph 的流式输出》相关文档 1', '《LangGraph 的流式输出》相关文档 2'], 'draft': '关于「LangGraph 的流式输出」：\n- 《LangGraph 的流式输出》相关文档 1\n- 《LangGraph 的流式输出》相关文档 2\n- 《LangGraph 的流式输出》相关文档 1\n- 《LangGraph 的流式输出》相关文档 2', 'approved': True, 'trail': ['start', 'rewrite', 'start', 'publish']}

approved = True
trail    = ['start', 'rewrite', 'start', 'publish']


In [22]:
# 回看整条执行轨迹（顶层视角）
print("--- get_state_history ---")
for snap in app.get_state_history(app_config):
    step = snap.metadata.get("step")
    trail = snap.values.get("trail", [])
    print(f"  step={step:>2}  next={str(snap.next):<18} trail={trail}")

--- get_state_history ---
  step= 6  next=()                 trail=['start', 'rewrite', 'start', 'publish']
  step= 5  next=('publish',)       trail=['start', 'rewrite', 'start']
  step= 4  next=('writer',)        trail=['start', 'rewrite', 'start']
  step= 3  next=('start',)         trail=['start', 'rewrite']
  step= 2  next=('rewrite',)       trail=['start']
  step= 1  next=('writer',)        trail=['start']
  step= 0  next=('start',)         trail=[]
  step=-1  next=('__start__',)     trail=[]


### 小结

| 主题 | 要点 |
| --- | --- |
| 调用方式 | 编译后的图当节点（`add_node("x", subgraph)`）；未编译的 builder 会报 `TypeError` |
| 状态可见性 | 子图节点只看到**自己 schema 声明过**的键，父图私有字段传不进去 |
| 状态回传 | 子图节点回传的是**子图完整 state**；父图私有字段不会被带出来 |
| ⚠️ reducer 双算 | 共享累积键（`Annotated[..., operator.add]`）会被父子各归约一次，值重复。**共享键优先用覆盖语义**，或按方案 A/C 隔离 |
| checkpointer | 只需父图挂；子图自动用同一个 saver，写进独立 `checkpoint_ns`（多层用 `\|` 拼接） |
| 子图自带 saver | 会被父图覆盖而失效，除非用「节点内手动 invoke」 |
| 中断 | 子图内 `interrupt()` 能被父图 `get_state` / `Command(resume=...)` 接住，父图视角只看到那个子图节点 |
| 静态断点 | `interrupt_before` 要么写父图这层的节点名，要么在**子图自己 compile 时**写死 |
| 排查手段 | `subgraphs=True` 看子图内每一步；带 `checkpoint_ns` 查子图状态 |

**最该记住的一条**：共享键优先用「覆盖」语义。累积类的 `log` / `history` 这类字段，让它在需要的那一层私有化，既避免双算，又让数据流向更清楚。